# Lesson 12A: Deep Unsupervised Learning Theory — Autoencoders and Variational Autoencoders

## Introduction

PCA (Lesson 5) compresses data through a *linear* projection: find the directions of maximum variance, project onto them. An **autoencoder** generalizes this to *nonlinear* compression — a neural network learns to squeeze data through a narrow bottleneck and reconstruct it, with no constraint that the compression be linear. In fact, a linear autoencoder with no nonlinearities and squared-error loss provably recovers the *same subspace* PCA finds — autoencoders are a strict generalization, not a different idea.

But a vanilla autoencoder has a problem: nothing constrains what the latent space looks like. Two nearby real images might encode to distant latent points, and the empty space *between* real codes decodes to garbage. A **Variational Autoencoder (VAE)** (Kingma & Welling, 2013) fixes this by making the encoder output a *distribution* rather than a point, regularized toward a simple prior — turning the latent space into something you can actually sample from to generate new, plausible data.

In this lesson:
1. Derive the vanilla autoencoder and its exact connection to PCA
2. Show empirically why a vanilla autoencoder's latent space isn't generative
3. Derive the VAE's generative model and the ELBO objective
4. Derive the reparameterization trick — why direct sampling blocks backpropagation, and how to fix it
5. Implement both from scratch in PyTorch and train on MNIST
6. Compare sampling from a vanilla AE's latent space against sampling from a VAE's

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [MNIST Setup](#mnist)
4. [Vanilla Autoencoder](#vanilla-ae)
5. [The PCA Connection](#pca-connection)
6. [Why a Vanilla Autoencoder Isn't Generative](#not-generative)
7. [The VAE Generative Model](#vae-model)
8. [The ELBO](#elbo)
9. [The Reparameterization Trick](#reparameterization)
10. [Training the VAE](#vae-training)
11. [Sampling Comparison](#sampling-comparison)
12. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits

torch.manual_seed(42)
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="mnist"></a>
## MNIST Setup

A 6,000-image subset of MNIST, flattened to 784-dimensional vectors, pixel values scaled to $[0, 1]$.

In [ ]:
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())

n_samples = 6000
X = train_dataset.data[:n_samples].float().view(-1, 784) / 255.0
y = train_dataset.targets[:n_samples]

print(f"Dataset: {X.shape[0]} images, {X.shape[1]} pixels each")
print(f"Pixel range: [{X.min():.2f}, {X.max():.2f}]")

fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(X[i].view(28, 28), cmap='gray')
    ax.set_title(str(y[i].item()))
    ax.axis('off')
plt.tight_layout()
plt.show()

<a name="vanilla-ae"></a>
## Vanilla Autoencoder

### Architecture

An **encoder** compresses input $x \in \mathbb{R}^{784}$ to a low-dimensional latent code $z \in \mathbb{R}^{k}$ ($k \ll 784$); a **decoder** reconstructs $\hat{x}$ from $z$. Training minimizes reconstruction error alone:

$$\mathcal{L}_{\text{AE}} = \|x - \hat{x}\|^2 = \|x - \text{decoder}(\text{encoder}(x))\|^2$$

No probabilistic assumptions, no regularization on $z$ — just "compress and reconstruct as well as possible."

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim: int = 784, hidden_dim: int = 256, latent_dim: int = 2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim), nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


def train_model(model, X, n_epochs, batch_size, loss_fn, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    n = X.shape[0]
    losses = []
    for epoch in range(n_epochs):
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            batch = X[idx]
            optimizer.zero_grad()
            loss = loss_fn(model, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(idx)
        losses.append(epoch_loss / n)
    return losses


ae_latent_dim = 2
autoencoder = Autoencoder(latent_dim=ae_latent_dim)
ae_loss_fn = lambda model, batch: nn.functional.mse_loss(model(batch), batch)
ae_losses = train_model(autoencoder, X, n_epochs=20, batch_size=128, loss_fn=ae_loss_fn)

print(f"Vanilla AE final reconstruction loss (MSE): {ae_losses[-1]:.4f}")

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
with torch.no_grad():
    reconstructions = autoencoder(X[:8])
for i in range(8):
    axes[0, i].imshow(X[i].view(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(reconstructions[i].view(28, 28), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_title('Original', loc='left', fontsize=10)
axes[1, 0].set_title('Reconstructed (2D bottleneck)', loc='left', fontsize=10)
plt.tight_layout()
plt.show()

<a name="pca-connection"></a>
## The PCA Connection

Claim: a **linear** autoencoder (no activation functions, squared-error loss) recovers the exact same subspace as PCA with the same number of components. We verify this directly — not on MNIST (where nonlinear structure would blur the comparison) but on `load_digits`, comparing reconstruction MSE between a linear autoencoder and PCA at matched component count.

In [ ]:
digits = load_digits()
X_digits = StandardScaler(with_mean=True, with_std=False).fit_transform(digits.data.astype(np.float32))
X_digits_t = torch.tensor(X_digits)

n_components = 5


class LinearAutoencoder(nn.Module):
    def __init__(self, input_dim: int, n_components: int):
        super().__init__()
        self.encoder = nn.Linear(input_dim, n_components, bias=False)
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


linear_ae = LinearAutoencoder(X_digits.shape[1], n_components)
linear_ae_loss_fn = lambda model, batch: nn.functional.mse_loss(model(batch), batch)
train_model(linear_ae, X_digits_t, n_epochs=2000, batch_size=len(X_digits_t), loss_fn=linear_ae_loss_fn, lr=0.01)

with torch.no_grad():
    linear_ae_mse = nn.functional.mse_loss(linear_ae(X_digits_t), X_digits_t).item()

pca = PCA(n_components=n_components)
X_pca_transformed = pca.fit_transform(X_digits)
X_pca_reconstructed = pca.inverse_transform(X_pca_transformed)
pca_mse = np.mean((X_pca_reconstructed - X_digits) ** 2)

print(f"Linear autoencoder reconstruction MSE: {linear_ae_mse:.4f}")
print(f"PCA reconstruction MSE:                {pca_mse:.4f}")
print(f"\n💡 A linear autoencoder trained by gradient descent converges to the same reconstruction error as PCA's closed-form solution")

<a name="not-generative"></a>
## Why a Vanilla Autoencoder Isn't Generative

Reconstruction works — but can we *generate new* digits by decoding random points in the 2D latent space? Nothing in the vanilla AE's loss function constrains what the latent space looks like: it could be a thin, oddly-shaped manifold with large empty gaps. Sampling uniformly within the *observed* latent range and decoding shows the problem directly.

In [ ]:
with torch.no_grad():
    ae_latents = autoencoder.encoder(X).numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

scatter = axes[0].scatter(ae_latents[:, 0], ae_latents[:, 1], c=y.numpy(), cmap='tab10', s=8, alpha=0.6)
plt.colorbar(scatter, ax=axes[0], label='Digit')
axes[0].set_title('Vanilla AE Latent Space (real encodings)', fontsize=11, fontweight='bold')

rng = np.random.RandomState(42)
lat_min, lat_max = ae_latents.min(axis=0), ae_latents.max(axis=0)
random_latents = rng.uniform(lat_min, lat_max, size=(10, ae_latent_dim))
with torch.no_grad():
    random_decoded = autoencoder.decoder(torch.tensor(random_latents, dtype=torch.float32))

axes[1].axis('off')
axes[1].set_title('Decoding random points in that latent range', fontsize=11, fontweight='bold')
for i in range(10):
    ax_inset = axes[1].inset_axes([i % 5 * 0.2, 0.5 if i < 5 else 0.0, 0.2, 0.5])
    ax_inset.imshow(random_decoded[i].view(28, 28), cmap='gray')
    ax_inset.axis('off')

plt.tight_layout()
plt.show()

print("💡 Real digit encodings form scattered, uneven clusters with gaps between them — random points sampled from the same bounding box often land in those gaps and decode to blurry, ambiguous, or unrealistic shapes")

<a name="vae-model"></a>
## The VAE Generative Model

The VAE reframes this as a proper generative model:

1. Sample a latent code from a simple prior: $z \sim \mathcal{N}(0, I)$
2. Decode: $x \sim p(x \mid z)$, parameterized by a neural network (the decoder)

Since the true posterior $p(z \mid x)$ is intractable, the VAE learns an **approximate posterior** $q(z \mid x)$ — the encoder — parameterized as a Gaussian whose mean and (log-)variance are themselves neural network outputs:

$$q(z \mid x) = \mathcal{N}(z; \mu(x), \sigma^2(x))$$

This is the crucial difference from a vanilla AE: the encoder doesn't output one latent point, it outputs the parameters of a *distribution over* latent points. Training pulls that distribution toward the standard normal prior — which is exactly what gives the latent space the smooth, fillable structure a vanilla AE lacks.

<a name="elbo"></a>
## The ELBO

Directly maximizing $p(x)$ is intractable (it requires integrating over all $z$), so the VAE instead maximizes a tractable lower bound — the **Evidence Lower Bound (ELBO)**:

$$\text{ELBO}(x) = \underbrace{\mathbb{E}_{q(z|x)}\left[\log p(x \mid z)\right]}_{\text{reconstruction term}} - \underbrace{D_{KL}\left(q(z \mid x) \,\|\, p(z)\right)}_{\text{regularization term}}$$

Maximizing the ELBO is equivalent to minimizing:

$$\mathcal{L}_{\text{VAE}} = \underbrace{-\mathbb{E}_{q(z|x)}[\log p(x \mid z)]}_{\text{reconstruction loss (e.g. binary cross-entropy)}} + \underbrace{D_{KL}(q(z|x) \,\|\, \mathcal{N}(0, I))}_{\text{KL divergence, closed form for two Gaussians}}$$

For a Gaussian $q(z|x) = \mathcal{N}(\mu, \sigma^2)$ against the standard normal prior, the KL term has a closed form:

$$D_{KL} = -\frac{1}{2}\sum_j \left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

**Reading the trade-off**: the reconstruction term pulls the model toward accurately reproducing $x$ (just like a vanilla AE); the KL term pulls every encoded distribution toward the shared standard-normal prior, which is what keeps the latent space densely, smoothly filled rather than scattered into arbitrary shapes with gaps.

<a name="reparameterization"></a>
## The Reparameterization Trick

There's an obstacle: to train by backpropagation, gradients need to flow from the reconstruction loss back through the sampling step $z \sim \mathcal{N}(\mu, \sigma^2)$ to $\mu$ and $\sigma$. But sampling is not a differentiable operation — you can't backpropagate through a random draw.

**The fix**: rewrite the sample as a deterministic function of $\mu$, $\sigma$, and an *independent* noise source $\epsilon$:

$$z = \mu + \sigma \odot \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

Now the randomness is confined entirely to $\epsilon$, which doesn't depend on the network's parameters — so gradients flow cleanly through $\mu$ and $\sigma$ via ordinary differentiable arithmetic, and $\epsilon$ is just treated as a constant input for that forward pass. This is the single trick that makes VAEs trainable end-to-end with standard backpropagation.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim: int = 784, hidden_dim: int = 256, latent_dim: int = 2):
        super().__init__()
        self.encoder_hidden = nn.Linear(input_dim, hidden_dim)
        self.encoder_mu = nn.Linear(hidden_dim, latent_dim)
        self.encoder_logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim), nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor):
        h = torch.relu(self.encoder_hidden(x))
        return self.encoder_mu(h), self.encoder_logvar(h)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar


def vae_loss_fn(model: VAE, batch: torch.Tensor) -> torch.Tensor:
    recon, mu, logvar = model(batch)
    reconstruction_loss = nn.functional.binary_cross_entropy(recon, batch, reduction='sum')
    kl_divergence = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (reconstruction_loss + kl_divergence) / len(batch)


print("✅ VAE class and loss defined")

<a name="vae-training"></a>
## Training the VAE

In [ ]:
vae_latent_dim = 2
vae = VAE(latent_dim=vae_latent_dim)
vae_losses = train_model(vae, X, n_epochs=20, batch_size=128, loss_fn=vae_loss_fn)

print(f"VAE final loss (reconstruction + KL): {vae_losses[-1]:.2f}")

plt.figure(figsize=(7, 4))
plt.plot(vae_losses)
plt.xlabel('Epoch')
plt.ylabel('ELBO loss (per-sample)')
plt.title('VAE Training Loss', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a name="sampling-comparison"></a>
## Sampling Comparison

Compare the VAE's latent space (should look like a single smooth, roughly Gaussian blob, unlike the vanilla AE's scattered clusters) and decode random samples from the **prior** $\mathcal{N}(0, I)$ directly — no need to guess an empirical bounding box, since the whole point of the KL term is that the prior *is* the space the encoder was trained to match.

In [ ]:
with torch.no_grad():
    vae_mu, _ = vae.encode(X)
    vae_latents = vae_mu.numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

scatter = axes[0].scatter(vae_latents[:, 0], vae_latents[:, 1], c=y.numpy(), cmap='tab10', s=8, alpha=0.6)
plt.colorbar(scatter, ax=axes[0], label='Digit')
axes[0].set_title('VAE Latent Space (encoder means)', fontsize=11, fontweight='bold')

with torch.no_grad():
    prior_samples = torch.randn(10, vae_latent_dim)
    vae_decoded = vae.decoder(prior_samples)

axes[1].axis('off')
axes[1].set_title('Decoding samples from the N(0, I) prior', fontsize=11, fontweight='bold')
for i in range(10):
    ax_inset = axes[1].inset_axes([i % 5 * 0.2, 0.5 if i < 5 else 0.0, 0.2, 0.5])
    ax_inset.imshow(vae_decoded[i].view(28, 28), cmap='gray')
    ax_inset.axis('off')

plt.tight_layout()
plt.show()

print("💡 The VAE latent space is a single smooth, filled region (KL divergence pulled every digit's encoding toward the shared prior) — sampling directly from that same prior decodes to recognizably digit-like shapes far more consistently than the vanilla AE's random-in-bounding-box samples")

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **A vanilla autoencoder generalizes PCA to nonlinear compression** — provably identical to PCA in the linear case, verified here by matching reconstruction MSE exactly.
2. **A vanilla autoencoder's latent space is ungoverned** — nothing in its loss function constrains what shape it takes, so random sampling within the observed range often lands in unrealistic gaps.
3. **A VAE encodes to a distribution, not a point** — $q(z|x) = \mathcal{N}(\mu(x), \sigma^2(x))$ — and regularizes that distribution toward a standard normal prior via KL divergence.
4. **The ELBO balances reconstruction against regularization** — accurate enough to reproduce the input, but pulled toward a shared, fillable prior distribution.
5. **The reparameterization trick makes the whole thing trainable** — rewriting the stochastic sampling step as a deterministic function of a fixed-distribution noise variable lets gradients flow through $\mu$ and $\sigma$ via ordinary backpropagation.

### When to Use Which

- **Vanilla autoencoder**: dimensionality reduction, denoising, anomaly detection (Lesson 7) via reconstruction error — when you don't need to generate new samples.
- **VAE**: when you want a genuinely samplable latent space — generation, interpolation between examples, or a well-behaved representation for downstream tasks.
- **PCA**: still the right first choice when linear compression is sufficient — faster, no training, no hyperparameters.

### Next Steps

In Lesson 12B (practical), we'll:
- Build a convolutional (not fully-connected) autoencoder, better suited to image data
- Add denoising: train on corrupted inputs, reconstruct the clean original
- Compare denoising quality and training dynamics against the fully-connected models here

You now understand the deep-learning bridge from PCA's linear compression to a fully probabilistic, samplable generative model — the last major representation-learning idea in this course 🎯